# STT 모델 비교 실습
## 실행 전 사전 준비

**가상환경 설치**
```bash
uv venv STT_env --python 3.12
STT_env\Scripts\activate
uv pip install -U qwen-asr
```

**ffmpeg 설치 (Whisper 오디오 파일 읽기에 필요)**
```bash
winget install ffmpeg
```
> 설치 후 VS Code 재시작 필요

## 1. 환경 확인

In [ ]:
import torch

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print(f"사용 디바이스: {device}")
print(f"데이터 타입 : {dtype}")
if torch.cuda.is_available():
    print(f"GPU 이름     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 오디오 경로 설정

In [ ]:
import os
from pathlib import Path

AUDIO_PATH = str(Path.cwd() / "6.01_김준테_멘토님_5min.mp3")

assert os.path.exists(AUDIO_PATH), f"파일을 찾을 수 없습니다: {AUDIO_PATH}"
print(f"오디오 파일 확인 완료: {AUDIO_PATH}")
print(f"파일 크기: {os.path.getsize(AUDIO_PATH) / 1e6:.2f} MB")

## 2-1. 오디오 청크 분할 (30초 / 2초 오버랩)

In [ ]:
import librosa
import numpy as np

audio_data, sample_rate = librosa.load(AUDIO_PATH, sr=None, mono=True)

CHUNK_SEC   = 30
OVERLAP_SEC = 2
STEP_SEC    = CHUNK_SEC - OVERLAP_SEC

chunk_samples = int(CHUNK_SEC   * sample_rate)
step_samples  = int(STEP_SEC    * sample_rate)

chunks = []
start = 0
while start < len(audio_data):
    end = min(start + chunk_samples, len(audio_data))
    chunks.append({
        "audio":     audio_data[start:end],
        "start_sec": start / sample_rate,
        "end_sec":   end   / sample_rate,
    })
    start += step_samples

print(f"샘플레이트 : {sample_rate} Hz")
print(f"전체 길이  : {len(audio_data) / sample_rate:.1f}초")
print(f"총 청크 수 : {len(chunks)}개 ({CHUNK_SEC}초 단위, {OVERLAP_SEC}초 오버랩)
")
for i, c in enumerate(chunks):
    print(f"  청크 {i+1:>3}: {c['start_sec']:6.1f}s ~ {c['end_sec']:6.1f}s  ({c['end_sec']-c['start_sec']:.1f}초)")

## 3. Whisper 모델로드

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
from transformers import pipeline


WHISPER_MODEL_PATH = Path.cwd() / "models" / "whisper-large-v3"
weight_file = WHISPER_MODEL_PATH / "model.safetensors"

if not weight_file.exists():
    print("Whisper 모델 weight 없음 → 다운로드 시작...")
    snapshot_download(repo_id="openai/whisper-large-v3", local_dir=WHISPER_MODEL_PATH)
    print("다운로드 완료")
else:
    print(f"Whisper 모델 확인 완료: {WHISPER_MODEL_PATH}")


whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=str(WHISPER_MODEL_PATH),
    dtype=dtype,
    device=device,
)

## 4. Whisper 추론 실행

In [ ]:
result = whisper_pipe(AUDIO_PATH, return_timestamps=True, generate_kwargs={"language": "korean"})
print(f"인식 결과:\n{result['text']}")

## 5. Qwen3 모델 로드

In [ ]:
from pathlib import Path
from huggingface_hub import snapshot_download
from qwen_asr import Qwen3ASRModel

MODEL_PATH = Path.cwd() / "models" / "Qwen3-ASR"

if not MODEL_PATH.exists():
    print("Qwen3 모델 없음 → 다운로드 시작...")
    snapshot_download(repo_id="Qwen/Qwen3-ASR-1.7B", local_dir=MODEL_PATH)
    print("다운로드 완료")

model = Qwen3ASRModel.from_pretrained(
    str(MODEL_PATH),
    dtype=dtype,
    device_map=device,
    max_inference_batch_size=8,  # 오디오를 청크로 나눠 병렬 처리할 배치 크기. 클수록 속도↑ but RAM 사용↑
    max_new_tokens=512,          # 한 번 추론에서 생성 가능한 최대 토큰 수. 짧으면 긴 발화가 중간에 잘림
)
print("모델 로드 완료")

## 6. Qwen3 추론 실행

In [ ]:
import tempfile, os

full_text_parts = []
with tempfile.TemporaryDirectory() as tmpdir:
    for i, chunk in enumerate(chunks):
        tmp_path = os.path.join(tmpdir, f"chunk_{i}.wav")
        sf.write(tmp_path, chunk["audio"], sample_rate)

        result = model.transcribe(audio=tmp_path, language="Korean")
        text = result[0].text.strip()
        full_text_parts.append(text)
        print(f"[청크 {i+1:>3}/{len(chunks)}] {chunk['start_sec']:6.1f}s ~ {chunk['end_sec']:6.1f}s | {text[:60]}...")

print(f"\n=== 전체 전사 결과 ===\n")
print(" ".join(full_text_parts))

## 7. Qwen3 컨텍스트 프롬프팅

`initial_prompt`에 도메인 키워드나 문맥 문장을 넣으면 모델이 해당 어휘를 우선적으로 인식한다.
- 자주 오인식되는 고유명사, 전문용어, 인명 등을 미리 힌트로 제공
- 길수록 좋은 게 아니라 핵심 단어 위주로 간결하게 작성하는 것이 효과적

In [ ]:
# 도메인에 맞는 키워드를 쉼표로 구분해 작성
# 출처: 2026-06-03.md — 고유명사·서비스명·참여자 이름 위주로 선별
CONTEXT_PROMPT = "Obsidian, LLM-Wiki, 클로버노트, STT, 멘토링, 화자 분리, 김상규, 박성준, 김건우, 홍사인, 이강혁, 김진기, 박세종, 정민기, 신동륜"

full_text_parts = []
with tempfile.TemporaryDirectory() as tmpdir:
    for i, chunk in enumerate(chunks):
        tmp_path = os.path.join(tmpdir, f"chunk_{i}.wav")
        sf.write(tmp_path, chunk["audio"], sample_rate)

        result = model.transcribe(audio=tmp_path, language="Korean", initial_prompt=CONTEXT_PROMPT)
        text = result[0].text.strip()
        full_text_parts.append(text)
        print(f"[청크 {i+1:>3}/{len(chunks)}] {chunk['start_sec']:6.1f}s ~ {chunk['end_sec']:6.1f}s | {text[:60]}...")

print(f"\n컨텍스트 프롬프트: {CONTEXT_PROMPT}")
print(f"\n=== 전체 전사 결과 ===\n")
print(" ".join(full_text_parts))

## 8. 화자 분리 + ASR (pyannote)

**사전 준비**
```
허깅스페이스 가입
```

**HuggingFace 토큰 및 약관 동의 필요** (1회)
- https://hf.co/pyannote/speaker-diarization-3.1
- https://hf.co/pyannote/segmentation-3.0

> CPU에서도 동작하지만 실시간보다 느림 — GPU 환경 권장

In [ ]:
import json
import soundfile as sf
import numpy as np
import tempfile, os
from pathlib import Path
from huggingface_hub import snapshot_download, login
from pyannote.audio import Pipeline

with open("keys.json") as f:
    keys = json.load(f)

HF_TOKEN = keys["hf_token"]
login(token=HF_TOKEN, add_to_git_credential=False)

DIARIZE_MODEL_PATH = Path.cwd() / "models" / "pyannote-diarization"
SEGMENT_MODEL_PATH = Path.cwd() / "models" / "pyannote-segmentation"

if not DIARIZE_MODEL_PATH.exists():
    print("화자 분리 모델 없음 → 다운로드 시작...")
    snapshot_download(
        repo_id="pyannote/speaker-diarization-3.1",
        local_dir=DIARIZE_MODEL_PATH,
    )
    snapshot_download(
        repo_id="pyannote/segmentation-3.0",
        local_dir=SEGMENT_MODEL_PATH,
    )
    print("다운로드 완료")
else:
    print(f"화자 분리 모델 확인 완료: {DIARIZE_MODEL_PATH}")

diarize_pipeline = Pipeline.from_pretrained(str(DIARIZE_MODEL_PATH))

# 묵음 구간이 1.0초 이하면 발화가 끊기지 않은 것으로 간주해 앞뒤 구간을 하나로 합침
# 값을 줄이면 짧은 침묵도 화자 전환으로 분리되고, 늘리면 긴 침묵까지 같은 발화로 묶임
diarize_pipeline.segmentation.min_duration_off = 1.0

diarize_pipeline = diarize_pipeline.to(torch.device(device))
print("화자 분리 파이프라인 로드 완료")

In [ ]:
import librosa
audio_data, sample_rate = librosa.load(AUDIO_PATH, sr=None, mono=True)

waveform = torch.tensor(audio_data).unsqueeze(0).float()
diarize_result = diarize_pipeline({"waveform": waveform, "sample_rate": sample_rate})

segments = []
for turn, _, speaker in diarize_result.itertracks(yield_label=True):
    segments.append({"speaker": speaker, "start": turn.start, "end": turn.end})

for seg in segments:
    print(f"[{seg['speaker']}] {seg['start']:.2f}s ~ {seg['end']:.2f}s")

print(f"
총 {len(segments)}개 구간, {len({s['speaker'] for s in segments})}명 화자 감지")

In [ ]:
# --- Whisper ---
print("=== Whisper 화자별 전사 ===\n")
with tempfile.TemporaryDirectory() as tmpdir:
    for i, seg in enumerate(segments):
        chunk = audio_data[int(seg["start"] * sample_rate):int(seg["end"] * sample_rate)]
        if len(chunk) == 0:
            continue
        tmp_path = os.path.join(tmpdir, f"seg_{i}.wav")
        sf.write(tmp_path, chunk, sample_rate)

        result = whisper_pipe(tmp_path, generate_kwargs={"language": "korean"})
        print(f"[{seg['speaker']}] {seg['start']:.1f}s ~ {seg['end']:.1f}s")
        print(f"  {result['text'].strip()}\n")

# --- Qwen3 ---
print("=== Qwen3 화자별 전사 ===\n")
with tempfile.TemporaryDirectory() as tmpdir:
    for i, seg in enumerate(segments):
        chunk = audio_data[int(seg["start"] * sample_rate):int(seg["end"] * sample_rate)]
        if len(chunk) == 0:
            continue
        tmp_path = os.path.join(tmpdir, f"seg_{i}.wav")
        sf.write(tmp_path, chunk, sample_rate)

        results = model.transcribe(audio=tmp_path, language="Korean")
        print(f"[{seg['speaker']}] {seg['start']:.1f}s ~ {seg['end']:.1f}s")
        print(f"  {results[0].text.strip()}\n")

## To Do. 
### 1. 녹음 파일 전사 결과를 파일로 저장
### 2. 클로드 or 코덱스 회의록 요약 스킬 만들기
### 3. 회의록 요약